# 03 – Werkzeugbasierte Erzeugung und Stirnschnitt-Zahnkontur

**Ziel.** Dieses Notebook dokumentiert, was ein zahnstangenförmiges Werkzeug (Wälzfräser, Hobelkamm, Schleifscheibe) am Außenrad erzeugt: Erzeugungs-Profilverschiebung aus den Zahndickenabmaßen, erzeugter Fußkreis, Fuß-Formkreis mit und ohne Unterschnitt, Kopf-Formkreis aus dem Kantenbrechwinkel des Werkzeugs, Kopfzahndicke und Restzahndicke, Zahnhöhen, Kopfspiel und Formübermaß – und die daraus abgetastete Zahnkontur im Stirnschnitt, gerad- und schrägverzahnt. Es sichert die Module `gearcore.generation`, `gearcore.trochoid` und `gearcore.contour` sowie den Erzeugungsvergleich in `gearcore.parity` ab.

**Regeln.** Das Notebook ist ausführbare Dokumentation (ADR-104): Es ruft `gearcore` auf und definiert selbst keine Formeln. Jede Aussage trägt Norm, Ausgabe, Abschnitt, Gleichung und Seite. Normtext wird nicht wörtlich wiedergegeben; die Formeln sind eigene Transkriptionen in der Schreibweise der aktuell gültigen Norm. Programmnamen, Formelzeichen und Benennungen stehen in der Größen-Registry (`data/quantities.yaml`). Gerechnet wird durchgängig mit doppelter Genauigkeit.

**Quellen.**

| Schlüssel in `sources.yaml` | Dokument | Inhalt in diesem Notebook |
|---|---|---|
| `ISO21771:2014` | DIN ISO 21771:2014-08 | §4.6 Höhen, §4.7 Zahndicke mit $x_{\mathrm{E}}$, §6.1 Kopfkantenbruch, §7.1 Werkzeug-Bezugsprofil, §7.3 Zahndickenabmaße, §7.4 Erzeugungs-Profilverschiebung, §7.5 erzeugter Fußkreis, §7.6 Formkreise (Gl. (128) in der Fassung des Anhangs NB), §7.7 Unterschnitt, §7.9 Kopfzahndicke |
| `DIN3960:1987` | DIN 3960:1987-03 (zurückgezogen) | Anhang A.3.1: Kopf-Formkreis und Restzahndicke bei Kantenbruch durch das Werkzeug (die aktuelle Norm gibt dafür keine Gleichung); Gegenprobe §3.6 |
| `FVA604-I:2012` | FVA 604 I, Flankengenerator (Heft 1017) | Verzahnungsgesetz (§3.1), Fußkurve als Hüllkurve der Werkzeug-Kopfrundung (S. 28), Werkzeug im Stirnschnitt: Ellipse (S. 29), Gl. (4.35)–(4.37) |
| `Linke2010` | Linke, Stirnradverzahnung, 2. Aufl. | Verzahnungsgesetz (§1.3), Fußkurve (§2.3.2) |
| `ISOTR6336-30:2022` | ISO/TR 6336-30:2022 | Rechenbeispiel 1 (Anhang A): $d_{\mathrm{fE}}$ und $d_{\mathrm{Ff}}$ aus dem gedruckten $x_{\mathrm{E}}$ |
| STplus 11.1F | Handbuch und Referenzläufe | Orakel für Zahlen und Konturen; Vorbelegungen (`data/stplus_program`) |

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import gearcore
from gearcore import contour as ct
from gearcore import generation as gn
from gearcore import involute as iv
from gearcore import trochoid as tr
from gearcore.data import (
    has_stplus,
    load_stplus,
    load_worked_example,
    printed_number,
    printed_tolerance,
    stplus_case_dirs,
)
from gearcore.errors import NotSupportedError
from gearcore.models.common import Pair
from gearcore.models.inputs import GearInput, PairInput
from gearcore.parity import (
    FORM_CIRCLE_ACCURACY_MM,
    ParityVerdict,
    compare_generation,
    compute_generation_from_data,
    detection_limits,
    generation_data,
)
from gearcore.rack import din3972_tool, iso53_basic_rack, tool_from_basic_rack
from gearcore.stplus_program import stplus_default, stplus_transverse_residual_tip_thickness
from gearcore.trace import equations_of

print(gearcore.__version__)

## 1 Werkzeug-Bezugsprofil und Erzeugungs-Profilverschiebung

**Normbezug.** DIN ISO 21771:2014-08: §7.1 Bild 35 und 36, S. 63–64 (Werkzeug-Bezugsprofil mit $h_{\mathrm{aP0}}$, $\rho_{\mathrm{aP0}}$, $h_{\mathrm{FfP0}}$, $\alpha_{\mathrm{kP}}$); §7.3 Gl. (118), (119), S. 65; §7.4 Gl. (120)–(124) und Bild 38, S. 67.

Das Werkzeug-Bezugsprofil ist das Gegenprofil zum Zylinderrad-Bezugsprofil (§7.1). Seine Kopfhöhe $h_{\mathrm{aP0}}$ erzeugt die Zahnfußhöhe, seine Kopfkantenrundung $\rho_{\mathrm{aP0}}$ die Fußausrundung. Der gerade Teil der Kopfflanke, $h_{\mathrm{FaP0}} = h_{\mathrm{aP0}} - \rho_{\mathrm{aP0}}\,(1 - \sin\alpha_{\mathrm{n}})$, ist die Klammer der Gl. (128) (Anhang NB, S. 6) und steht als Zeichen im Bild 36 a) und in der Legende zu Bild 39 (S. 69). Der Kantenbrechwinkel $\alpha_{\mathrm{kP}}$ ab der Höhe $h_{\mathrm{FfP0}}$ über der Profilbezugslinie beschreibt eine Kantenbrechflanke, die am Rad einen Kopfkantenbruch erzeugt (Abschnitt 4).

Die Zahndickenabmaße $E_{\mathrm{sns}}$, $E_{\mathrm{sni}}$ (§7.3) legen die Grenzmaße der Normalzahndicke fest; sie gehen in die Erzeugungs-Profilverschiebung ein:

$$s_{\mathrm{ns}} = s_{\mathrm{n}} + E_{\mathrm{sns}} \quad (118) \qquad s_{\mathrm{ni}} = s_{\mathrm{n}} + E_{\mathrm{sni}} \quad (119)$$

$$x_{\mathrm{Es}} = x + \frac{E_{\mathrm{sns}}}{2\, m_{\mathrm{n}} \tan\alpha_{\mathrm{n}}} \quad (123) \qquad x_{\mathrm{Ei}} = x + \frac{E_{\mathrm{sni}}}{2\, m_{\mathrm{n}} \tan\alpha_{\mathrm{n}}} \quad (124)$$

Für eine Vorverzahnung mit Bearbeitungszugabe $q$ verschiebt Gl. (120)/(121) den Erzeugungswert um $q / \sin\alpha_{\mathrm{n}}$; das Fertigverzahnwerkzeug, das die Zugabe abnimmt, ist ein Erweiterungspunkt (`finishing_tool`), ein Werkzeug mit $q > 0$ wird in diesem Inkrement nicht erzeugt (Abschnitt 7).

**Welcher Wert gilt.** Die Erzeugung wird mit $x_{\mathrm{Es}}$ ausgewertet (oberes Abmaß, größtes Zahnvolumen), wie STplus es druckt; $x_{\mathrm{Ei}}$ und der Fußkreis mit $x_{\mathrm{Ei}}$ ($d_{\mathrm{fEi}}$, Bild 38) werden mit ausgegeben. Ohne Abmaße gilt $x_{\mathrm{E}} = x$, und das Ergebnis sagt das.

**Befund (STplus).** STplus druckt als „oberes Zahndickenabmaß“ A_ste den **Stirnschnittwert** $E_{\mathrm{sns}} / \cos\beta$ (DIN 3967 Anhang A, S. 7, nennt A_ste so), obwohl das Passsystem der DIN 3967 im Normalschnitt definiert ist (§1, S. 1): Der Lauf `helix30_z25_40` druckt −0,098 mm, wo die Reihe c25 für $d$ = 86,6 mm −0,085 mm nennt. Mit dem Normalschnittwert $-0{,}098 \cos 30° = -0{,}085$ mm reproduziert Gl. (123) den von STplus gedruckten $x_{\mathrm{E}}$ genau.

In [ ]:
# Referenzfall fzg_c (FZG-C-Radsatz, z = 16/24, m_n = 4,5, Werkzeug h_aP0* = 1,39, rho_aP0* = 0,25):
# das Paar aus input.ste, die Abmaße aus der STplus-Schnittstellendatei (Normalschnitt, in um).
data = generation_data("fzg_c")
result = compute_generation_from_data(data, data.values)
listing = load_stplus("fzg_c", "geometry")

rows = []
for role, generated in zip(("pinion", "wheel"), result.gears.as_tuple(), strict=True):
    index = 0 if role == "pinion" else 1
    rows.append(
        {
            "Rad": role,
            "x (Nennwert)": generated.profile_shift_coefficient,
            "E_sns in um": data.values[f"E_sns{index + 1}"],
            "E_sni in um": data.values[f"E_sni{index + 1}"],
            "x_Es (verwendet)": generated.generating_profile_shift_coefficient,
            "x_Ei": generated.lower_generating_profile_shift_coefficient,
            "x_E STplus": listing["x_E"]["numbers"][index],
            "d_fE in mm": generated.generated_root_diameter_mm,
            "d_fEi in mm": generated.lower_generated_root_diameter_mm,
            "d_f STplus": listing["d_f"]["numbers"][index],
        }
    )
    assert abs(
        generated.generating_profile_shift_coefficient - listing["x_E"]["numbers"][index]
    ) <= printed_tolerance(4)
    assert abs(
        generated.generated_root_diameter_mm - listing["d_f"]["numbers"][index]
    ) <= printed_tolerance(3)
pd.DataFrame(rows)

In [ ]:
# Die Gleichungen, die die Funktionen tragen, sind abrufbar (Rückverfolgbarkeit, ADR-103).
for function in (
    gn.generating_profile_shift_coefficient,
    gn.pre_machining_generating_profile_shift_coefficient,
    gn.tool_tip_form_height,
):
    print(function.__name__, "->", "; ".join(str(ref) for ref in equations_of(function)))

## 2 Erzeugter Fußkreis, Fuß-Formkreis, Unterschnitt

**Normbezug.** DIN ISO 21771:2014-08: §7.5 Gl. (125), S. 68; §7.6 Gl. (128), S. 69, in der korrigierten Fassung des nationalen Anhangs NB, S. 6; Gl. (129), (130), S. 69; §7.7 Gl. (135), S. 70; §4.6 Gl. (35)–(37), S. 34. Gegenprobe DIN 3960:1987-03 Gl. (3.6.04), (3.6.06), S. 14; (3.6.08)–(3.6.10), S. 16.

$$d_{\mathrm{fE}} = d + 2\, x_{\mathrm{E}}\, m_{\mathrm{n}} - 2\, h_{\mathrm{aP0}} \quad (125)$$

$$d_{\mathrm{Ff}} = \sqrt{\left(d \sin\alpha_{\mathrm{t}} - \frac{2\,[h_{\mathrm{aP0}} - x_{\mathrm{E}}\, m_{\mathrm{n}} - \rho_{\mathrm{aP0}}\,(1 - \sin\alpha_{\mathrm{n}})]}{\sin\alpha_{\mathrm{t}}}\right)^2 + d_{\mathrm{b}}^2} \quad (128)$$

$$x_{\mathrm{Emin}} = \frac{h_{\mathrm{FaP0}}}{m_{\mathrm{n}}} - \frac{z \sin^2\alpha_{\mathrm{t}}}{2 \cos\beta} \quad (135)$$

Gl. (128) gilt für Räder ohne Unterschnitt: Die Klammer ist die Wälzlänge vom Berührpunkt $T$ der Eingriffslinie bis zu dem Punkt, den das Ende der geraden Werkzeugflanke erzeugt. Wird sie negativ, reicht die gerade Flanke über $T$ hinaus – das ist genau die Bedingung $x_{\mathrm{E}} < x_{\mathrm{Emin}}$ der Gl. (135) – und die Kopfrundung des Werkzeugs schneidet in die Evolvente. Der Fuß-Formkreis ist dann der Schnittpunkt der Fußkurve mit der Evolvente (§7.6, S. 70; DIN 3960 §3.6.7, S. 16: „erfordert Iterationsrechnungen“), Abschnitt 3.

Die Zahnhöhen der §4.6 werden mit dem erzeugten Fußkreis ausgewertet (erste Form der Gl. (35)–(37)), wie STplus sie druckt: $h = (d_{\mathrm{a}} - d_{\mathrm{fE}})/2$, $h_{\mathrm{a}} = (d_{\mathrm{a}} - d)/2$, $h_{\mathrm{f}} = (d - d_{\mathrm{fE}})/2$.

**Befund (Norm).** Gl. (130) druckt in der Klammer $(1 - \sin\alpha_{\mathrm{t}})$, Anhang NB korrigiert Gl. (128) auf $(1 - \sin\alpha_{\mathrm{n}})$, korrigiert Gl. (130) aber nicht. Für Geradverzahnung sind beide gleich; für das schrägverzahnte Beispiel 1 der ISO/TR 6336-30:2022 ($\beta$ = 15,8°) gibt die korrigierte Gl. (128) mit dem gedruckten $x_{\mathrm{E1}}$ = 0,117 79 den gedruckten Wert $d_{\mathrm{Ff1}}$ = 132,248 mm (S. 45), Gl. (130) in der gedruckten Form 132,243 mm. `gearcore` rechnet nach Anhang NB; Gl. (130) ist als Gegenprobe implementiert und zeigt den Unterschied. Die Höhe der Tangente an die Werkzeug-Kopfrundung ist eine Höhe im Normalschnitt und bleibt im Stirnschnitt gleich (FVA 604 I, S. 29), was für $\sin\alpha_{\mathrm{n}}$ spricht.

In [ ]:
example = load_worked_example("iso_tr_6336_30_2022_annex_a_example_1")
inputs, expected = example["inputs"], example["expected"]
alpha_n = math.radians(inputs["normal_pressure_angle_deg"]["value"])
beta = math.radians(inputs["helix_angle_deg"]["value"])
m_n = inputs["normal_module_mm"]["value"]
alpha_t = iv.transverse_pressure_angle(alpha_n, beta)

rows = []
for role in ("pinion", "wheel"):
    z = inputs["number_of_teeth"][role]
    x_E = expected["generating_profile_shift_coefficient"][
        role
    ]  # aus den Zahnweiten der Norm (S. 45)
    h_aP0 = inputs["basic_rack_dedendum_mm"][role]
    rho_aP0 = inputs["basic_rack_fillet_radius_mm"][role]
    d = iv.reference_diameter(z, m_n, beta)
    d_b = iv.base_diameter(z, m_n, alpha_n, beta)
    h_FaP0 = gn.tool_tip_form_height(h_aP0, rho_aP0, alpha_n)
    d_fE = gn.generated_root_diameter(d, x_E, m_n, h_aP0)
    d_Ff = gn.root_form_diameter(d, alpha_t, h_FaP0, x_E, m_n, d_b)
    d_Ff_130 = gn.root_form_diameter_by_roll_angle(d_b, z, alpha_t, beta, h_aP0, rho_aP0, m_n, x_E)
    for key, value in (("generated_root_diameter_mm", d_fE), ("root_form_diameter_mm", d_Ff)):
        printed, decimals = printed_number(expected[key]["printed"][role])
        assert abs(value - printed) <= printed_tolerance(decimals), (role, key)
    rows.append(
        {
            "Rad": role,
            "x_E (gedruckt)": x_E,
            "d_fE": d_fE,
            "d_fE Norm": expected["generated_root_diameter_mm"]["printed"][role],
            "d_Ff Gl. (128) NB": d_Ff,
            "d_Ff Norm": expected["root_form_diameter_mm"]["printed"][role],
            "d_Ff Gl. (130) wie gedruckt": d_Ff_130,
            "x_Emin": gn.min_generating_profile_shift_coefficient(h_FaP0, m_n, z, alpha_t, beta),
        }
    )
pd.DataFrame(rows)

## 3 Die Fußkurve als Hüllkurve der Werkzeug-Kopfrundung

**Quellen.** Verzahnungsgesetz: Linke 2010 §1.3, S. 41 (die gemeinsame Normale der Flanken geht in jedem Berührpunkt durch den Wälzpunkt C); FVA 604 I §3.1 Gl. (3.2), (3.3), S. 12–13. Fußkurve: FVA 604 I S. 28 (Parallelkurve der Trochoide des Kopfrundungsmittelpunkts bei Geradverzahnung) und S. 29 (Werkzeug im Stirnschnitt um $1/\cos\beta$ gestreckt: die Kopfrundung wird zur Ellipse; FVA 604 I nähert sie durch einen Kreis mit $\rho_{\mathrm{aP0}}/\cos\beta$); Linke 2010 §2.3.2 Bild 2.3/1, Gl. (2.3/11), (2.3/12), S. 111–112.

`gearcore.trochoid` übernimmt keine geschlossene Fußkurvenformel. Jeder Punkt des Werkzeugprofils erzeugt mit seiner Tangente denjenigen Punkt des Rades, in dessen Wälzstellung seine Normale durch den Wälzpunkt geht (Hüllbedingung). Auf die gerade Flanke angewandt ergibt das exakt die Evolvente (Test des Moduls), auf die Kopflinie den Fußkreis, auf die Ellipse der Kopfrundung die Fußkurve. Schneidet die Fußkurve in die Evolvente (Unterschnitt), wird der Schnittpunkt numerisch bestimmt (`brentq`); sie endet dann außerhalb der Evolvente auf deren gespiegeltem Ast jenseits von $T$ – das ist der Wert, den Gl. (128) mit negativer Klammer liefern würde.

**Befund (Ellipse).** Im Vergleich mit den STplus-Konturen (Abschnitt 5) trifft die Ellipse die Fußkurve bei $\beta$ = 30° auf 0,6 µm; die Kreisnäherung mit konstantem Radius $\rho_{\mathrm{aP0}}/\cos\beta$ um den Rundungsmittelpunkt läge um 176 µm daneben ($\beta$ = 20°: 96 µm, $\beta$ = 15°: 26,5 µm; `scripts/circle_approximation_fva604.py`). STplus rechnet also mit der exakten Stirnschnitt-Kinematik, und `gearcore` ebenso.

In [ ]:
def _plot_fillet(case: str, role: str, ax):
    # Fußkurve, Evolvente, Grundkreis und Fußformkreis eines Rades im Stirnschnitt (halbe Zahndicke als Winkel)
    data = generation_data(case)
    result = compute_generation_from_data(data, data.values)
    pair = result.inputs
    generated = result.gears.pinion if role == "pinion" else result.gears.wheel
    z = generated.number_of_teeth
    m_n = pair.normal_module_mm
    alpha_n = math.radians(pair.normal_pressure_angle_deg)
    beta = math.radians(pair.helix_angle_deg if role == "pinion" else -pair.helix_angle_deg)
    alpha_t = iv.transverse_pressure_angle(alpha_n, beta)
    d_b = iv.base_diameter(z, m_n, alpha_n, beta)
    x_E = generated.generating_profile_shift_coefficient
    psi_b = iv.base_tooth_thickness_half_angle(
        iv.tooth_thickness_half_angle(z, x_E, alpha_n), alpha_t
    )
    rounding = tr.tip_rounding(
        z, m_n, alpha_n, beta, generated.tool_addendum_mm, generated.tool_tip_radius_mm, x_E
    )
    theta, radius, psi = tr.fillet_curve(rounding, n=400)
    ax.plot(
        radius * np.sin(psi),
        radius * np.cos(psi),
        label="Fußkurve (Hüllkurve, ganzer Parameterbereich)",
    )
    r_inv = np.linspace(0.5 * d_b, 0.5 * generated.tip_form_diameter_mm, 200)
    psi_inv = [tr.involute_half_angle(float(r), d_b, psi_b) for r in r_inv]
    ax.plot(r_inv * np.sin(psi_inv), r_inv * np.cos(psi_inv), "--", label="Evolvente ab Grundkreis")
    for name, diameter, style in (
        ("d_b", d_b, ":"),
        ("d_Ff", generated.root_form_diameter_mm, "-."),
        ("d_fE", generated.generated_root_diameter_mm, ":"),
    ):
        angles = np.linspace(-0.05, 0.25, 50)
        ax.plot(
            0.5 * diameter * np.sin(angles),
            0.5 * diameter * np.cos(angles),
            style,
            lw=0.8,
            label=name,
        )
    ax.set_aspect("equal")
    x_Emin = generated.min_generating_profile_shift_coefficient
    ax.set_title(
        f"{case} {role}: Unterschnitt = {generated.undercut}, x_E = {x_E:.4f}, x_Emin = {x_Emin:.4f}"
    )
    ax.set_xlabel("x in mm")
    ax.set_ylabel("y in mm")
    ax.legend(fontsize=8)
    return generated


fig, axes = plt.subplots(1, 2, figsize=(14, 6))
undercut = _plot_fillet("undercut_z12_x0", "pinion", axes[0])
regular = _plot_fillet("helix30_z25_40", "pinion", axes[1])
assert undercut.undercut and not regular.undercut
plt.show()

In [ ]:
# Die Ellipse im Stirnschnitt: Halbachsen rho_aP0 / cos(beta) längs der Zahnstange und rho_aP0 radial
rounding = tr.tip_rounding(25, 3.0, math.radians(20.0), math.radians(30.0), 3.75, 1.14, 0.3)
print(
    "Halbachse längs der Zahnstange:",
    rounding.semi_axis_xi_mm,
    "mm; radial:",
    rounding.semi_axis_eta_mm,
    "mm",
)
print(
    "Parameter des Übergangs zur geraden Flanke:",
    math.degrees(rounding.flank_parameter_rad),
    "deg (= -alpha_n)",
)
print("Fußkreisradius:", rounding.root_radius_mm, "mm")

## 4 Kopf: Kantenbruch durch das Werkzeug, Kopfzahndicke, Restzahndicke

**Normbezug.** DIN ISO 21771:2014-08 §7.6 Gl. (127), S. 68 ($d_{\mathrm{Fa}} = d_{\mathrm{a}} - 2 h_{\mathrm{K}}$); §6.1.2 und Bild 24, S. 55 (ein Kopfkantenbruch wird durch $h_{\mathrm{K}}$ und $s_{\mathrm{aK}}$ beschrieben); §7.9, S. 70 (Kopfzahndicke nach Gl. (38)); §4.7.5 Gl. (48), S. 37. Für den Kantenbruch, den die Kantenbrechflanke des Werkzeugs erzeugt, gibt die aktuelle Norm keine Gleichung; sie steht in DIN 3960:1987-03 Anhang A.3.1, S. 52:

$$s_{\mathrm{tK}} = \frac{m_{\mathrm{t}} \pi}{2} + 2\, h_{\mathrm{FfP0}}\,(\tan\alpha_{\mathrm{tK}} - \tan\alpha_{\mathrm{t}}) + 2\, x_{\mathrm{E}}\, m_{\mathrm{n}} \tan\alpha_{\mathrm{tK}} = d\,(\psi_{\mathrm{bK}} - \operatorname{inv}\alpha_{\mathrm{tK}}) \quad \text{(A.3.03)}$$

$$\psi_{\mathrm{bK}} - \psi_{\mathrm{b}} - \operatorname{inv}\alpha_{\mathrm{tKFa}} + \operatorname{inv}\alpha_{\mathrm{tFa}} = 0 \quad \text{(A.3.06)} \qquad s_{\mathrm{aK}} = d_{\mathrm{a}}\,(\psi_{\mathrm{bK}} - \operatorname{inv}\alpha_{\mathrm{taK}}) \quad \text{(A.3.05)}$$

Die Kantenbrechflanke ist eine zweite Zahnstangenflanke mit dem Winkel $\alpha_{\mathrm{kP}}$ (im Stirnschnitt $\alpha_{\mathrm{tK}}$ wie in Gl. (14)); sie erzeugt eine zweite Evolvente mit dem Grundkreis $d_{\mathrm{bK}} = d \cos\alpha_{\mathrm{tK}}$. Der Kopf-Formkreis ist der Durchmesser, auf dem beide Evolventen gleich dick sind (A.3.06, Iteration zwischen $d_{\mathrm{b}}$ und $d_{\mathrm{a}}$); erreicht die Kantenbrechflanke den Kopfkreis nicht, ist $d_{\mathrm{Fa}} = d_{\mathrm{a}}$. Die kurze Trochoide der Profilecke des Werkzeugs zwischen beiden Evolventen wird wie in DIN 3960 nicht modelliert.

**Eingegebener Kantenbruch.** Ein durch $h_{\mathrm{K}}$ gegebener Kantenbruch braucht seine Restzahndicke $s_{\mathrm{aK}}$ (Bild 24). STplus belegt stattdessen den Tangentialbetrag mit $0{,}7\, h_{\mathrm{K}}$ vor (Handbuch S. 19 und S. 225, Einstellgröße `TANG_BETRAG_ZU_H_KGF`), rechnet die Restdicke im Normalschnitt, $s_{\mathrm{aK}} = s_{\mathrm{an}} - 2 \cdot 0{,}7\, h_{\mathrm{K}}$, mindestens $0{,}2\, s_{\mathrm{an}}$, und begrenzt $h_{\mathrm{K}}$ auf $0{,}20\, m_{\mathrm{n}}$ (Probeläufe in `data/stplus_program/probes`). Der Rechenkern übernimmt das nicht: Ohne $s_{\mathrm{aK}}$ meldet das Ergebnis `tip_chamfer_shape_not_given`, und die Kontur verlangt den Wert. Der `.ste`-Importer liest eine Datei dagegen so, wie STplus sie rechnet, setzt $s_{\mathrm{aK}}$ nach dieser Regel und vermerkt es (ADR-114, Abschnitt 8).

In [ ]:
# kst-C (Rad: Kantenbrechwinkel 45 deg ab h_FfP0* = 0,6973, rho_aP0* = 0,33), eigener Neulauf 11.1F
data = generation_data("kst_c_rerun")
result = compute_generation_from_data(data, data.values)
listing = load_stplus("kst_c_rerun", "geometry")
rest = listing["Restdicke bei Kantenbruch/Kopfruecknahme"]["numbers"]
rows = []
for index, (role, generated) in enumerate(
    zip(("pinion", "wheel"), result.gears.as_tuple(), strict=True)
):
    rows.append(
        {
            "Rad": role,
            "alpha_kP in deg": generated.tool_edge_break_angle_deg,
            "h_FfP0 in mm": generated.tool_root_form_height_mm,
            "d_a": generated.tip_diameter_mm,
            "d_Fa": generated.tip_form_diameter_mm,
            "d_Fa STplus": listing["d_Fa"]["numbers"][index],
            "h_K": generated.tip_chamfer_radial_mm,
            "h_K STplus": listing["h_K"]["numbers"][index],
            "s_at (Gl. (38))": generated.transverse_tip_tooth_thickness_mm,
            "s_an (Gl. (48))": generated.normal_tip_tooth_thickness_mm,
            "s_an STplus": listing["s_an"]["numbers"][index],
            "s_aK (A.3.05)": generated.residual_tip_thickness_mm,
            "Restdicke STplus": rest[index],
            "Hinweise": ", ".join(w.code for w in generated.warnings),
        }
    )
wheel = result.gears.wheel
assert (
    wheel.tool_edge_break_angle_deg == 45.0 and wheel.tip_form_diameter_mm < wheel.tip_diameter_mm
)
assert abs(wheel.tip_form_diameter_mm - listing["d_Fa"]["numbers"][1]) <= FORM_CIRCLE_ACCURACY_MM
assert abs(wheel.residual_tip_thickness_mm - rest[1]) <= printed_tolerance(3)
pd.DataFrame(rows)

## 5 Zahnkontur im Stirnschnitt und Vergleich mit STplus

`gearcore.contour.tooth_contour` setzt einen Zahn aus den erzeugten Elementen zusammen: Fußkurve vom Fußkreis bis zum Fuß-Formkreis, Evolvente bis zum Kopf-Formkreis, Kantenbruch-Evolvente oder gegebener Kantenbruch bis zum Kopfkreis, Kopfkreisbogen; ein Zahn auf der +y-Achse vom Fußkreis der linken Flanke über den Kopf zum Fußkreis der rechten Flanke, wie der STplus-Konturexport (Handbuch §6.4.2). `gear_polygon` wiederholt den Zahn $z$-mal mit dem Fußkreis dazwischen. `compare` misst den Normalabstand von Referenzpunkten zur Kontur je Bereich (Fuß, Evolvente, Kopf).

**Gate des Plans:** ≤ 5 µm auf der Evolvente, ≤ 10 µm im Fuß, gerad- und schrägverzahnt. Erreicht werden auf allen 14 eigenen Läufen (28 Räder, darunter vier mit Unterschnitt und $\beta$ bis 30°) unter 0,6 µm im Fuß und unter 0,25 µm auf der Evolvente; die Tests pinnen diese Werte.

In [ ]:
def _with_stplus_chamfer(pair, result):
    # ein nur durch h_K gegebener Kantenbruch bekommt s_aK aus der STplus-Vorbelegung (0,7 h_K tangential), ausdrücklich
    gears = []
    for gear, generated in zip(pair.gears.as_tuple(), result.gears.as_tuple(), strict=True):
        if gear.tip_chamfer_radial_mm > 0.0 and gear.residual_tip_thickness_mm is None:
            s_aK = stplus_transverse_residual_tip_thickness(
                generated.transverse_tip_tooth_thickness_mm,
                generated.normal_tip_tooth_thickness_mm,
                gear.tip_chamfer_radial_mm,
            )
            gears.append(gear.model_copy(update={"residual_tip_thickness_mm": s_aK}))
        else:
            gears.append(gear)
    pair = pair.model_copy(update={"gears": Pair(pinion=gears[0], wheel=gears[1])})
    return gn.compute_generation(pair)


print("STplus-Vorbelegung:", stplus_default("tip_chamfer_tangential").rule)

In [ ]:
diffs = []
contours = {}
for path in stplus_case_dirs():
    case = path.name
    data = generation_data(case)
    result = compute_generation_from_data(data, data.values)
    result = _with_stplus_chamfer(result.inputs, result)
    pair = result.inputs
    for gear in (1, 2):
        if not has_stplus(case, f"contour_wz{gear}"):
            continue
        role = "pinion" if gear == 1 else "wheel"
        ours = ct.tooth_contour(result, role)
        diff = ct.compare(ours, ct.stplus_contour(case, gear))
        contours[(case, role)] = (ours, ct.stplus_contour(case, gear))
        generated = result.gears.pinion if role == "pinion" else result.gears.wheel
        diffs.append(
            {
                "Fall": case,
                "Rad": role,
                "beta in deg": pair.helix_angle_deg,
                "Unterschnitt": generated.undercut,
                "Kantenbruch": generated.tip_form_diameter_mm < generated.tip_diameter_mm,
                "Punkte STplus": diff.reference_points,
                "Fuß max in um": diff.fillet_max_um,
                "Evolvente max in um": diff.involute_max_um,
                "Kopf max in um": diff.tip_max_um,
                "gesamt max in um": diff.max_um,
            }
        )
table = pd.DataFrame(diffs)
assert table["Evolvente max in um"].max() <= 5.0 and table["Fuß max in um"].max() <= 10.0
assert table["Evolvente max in um"].max() <= 0.25 and table["Fuß max in um"].max() <= 0.6
table.style.format(
    {
        "Fuß max in um": "{:.3f}",
        "Evolvente max in um": "{:.3f}",
        "Kopf max in um": "{:.3f}",
        "gesamt max in um": "{:.3f}",
    }
)

In [ ]:
def _plot_overlay(case: str, role: str, ax):
    ours, theirs = contours[(case, role)]
    points = ours.as_array()
    ax.plot(points[:, 0], points[:, 1], lw=1.0, label="gearcore")
    ax.plot(theirs[:, 0], theirs[:, 1], ".", ms=2.5, label="STplus (Konturexport)")
    for name in ("left_fillet", "right_fillet"):
        segment = ours.segment(name)
        ax.plot(segment[:, 0], segment[:, 1], lw=2.0, color="C3")
    ax.set_aspect("equal")
    ax.set_title(f"{case} {role} (Unterschnitt = {ours.undercut})")
    ax.set_xlabel("x in mm")
    ax.set_ylabel("y in mm")
    ax.legend(fontsize=8)


def _plot_deviation(case: str, role: str, ax):
    ours, theirs = contours[(case, role)]
    diff = ct.compare(ours, theirs)
    radius = np.hypot(theirs[:, 0], theirs[:, 1])
    distance = ct.distances(ours, theirs)  # um, the measurement of the package
    ax.plot(2.0 * radius, distance, ".", ms=3)
    for name, value in (("d_Ff", ours.root_form_diameter_mm), ("d_Fa", ours.tip_form_diameter_mm)):
        ax.axvline(value, color="C3", lw=0.8, ls="--")
        ax.text(value, distance.max(), name, fontsize=8)
    ax.set_xlabel("Durchmesser des STplus-Punkts in mm")
    ax.set_ylabel("Normalabstand zur gearcore-Kontur in um")
    ax.set_title(f"{case} {role}: max {diff.max_um:.3f} um ({diff.region_of_max})")


fig, axes = plt.subplots(2, 2, figsize=(14, 12))
_plot_overlay("fzg_c", "pinion", axes[0, 0])
_plot_deviation("fzg_c", "pinion", axes[1, 0])
_plot_overlay("helix30_z25_40", "wheel", axes[0, 1])
_plot_deviation("helix30_z25_40", "wheel", axes[1, 1])
plt.tight_layout()
plt.show()

In [ ]:
def _plot_gear(contour, ax):
    polygon = ct.gear_polygon(contour, arc_points=6)
    closed = np.vstack((polygon, polygon[:1]))
    ax.plot(closed[:, 0], closed[:, 1], lw=0.8)
    ax.set_aspect("equal")
    ax.set_title(f"z = {contour.number_of_teeth}, {len(polygon)} Punkte")
    ax.set_xlabel("x in mm")
    ax.set_ylabel("y in mm")


fig, ax = plt.subplots(figsize=(7, 7))
_plot_gear(contours[("undercut_z12_x0", "pinion")][0], ax)
plt.show()

## 6 Parität der Erzeugungsgrößen mit STplus

Der Vergleich (`gearcore.parity.compare_generation`) rechnet jeden der 18 Fälle aus dem Paar der Eingabedatei mit den Abmaßen, die STplus gedruckt hat (in den Normalschnitt umgerechnet), und vergleicht $x_{\mathrm{E}}$, $d_{\mathrm{fE}}$, $d_{\mathrm{Ff}}$, $d_{\mathrm{Fa}}$, $h_{\mathrm{K}}$ und $s_{\mathrm{aK}}$ (bei Kantenbruch durch das Werkzeug), $s_{\mathrm{an}}$, $h$, $h_{\mathrm{a}}$, das Kopfspiel $c$ (Gl. (60)) und das Formübermaß $c_{\mathrm{F}}$ (Gl. (76)) mit Listing und Schnittstellendatei. Toleranzen wie im Paarvergleich (Notebook 02): halbe Einheit der letzten gedruckten Stelle, Fehlerfortpflanzung der einfachen Genauigkeit von STplus, Rundung gedruckter Eingaben.

**Befund (STplus, Formkreise).** STplus bestimmt den Fuß-Formkreis nicht nach Gl. (128), sondern als numerischen Übergang zweier Kurven: Sein $d_{\mathrm{Ff}}/2$ ist auf allen eigenen Läufen der Radius eines eigens gesetzten, doppelten Konturpunkts, während seine Kurven mit denen von `gearcore` auf 0,6 µm übereinstimmen. Ein tangentialer Übergang, der mit einer Normalentoleranz $\varepsilon$ gefunden wird, liegt längs der Kurven innerhalb $\sqrt{2\varepsilon/\Delta\kappa}$; Zehntel Mikrometer werden so zu Mikrometern. Gemessen gegen Gl. (128) bzw. den exakten Schnittpunkt: bis 6,4 µm über die 36 Räder. Der Vergleich räumt dem STplus-Wert diese Genauigkeit ein (`FORM_CIRCLE_ACCURACY_MM`, halb für $c_{\mathrm{F}}$ und $h_{\mathrm{K}}$); die gemessenen Maxima sind in `test_stplus_parity` gepinnt.

In [ ]:
rows = [row for path in stplus_case_dirs() for row in compare_generation(path.name)]
table = pd.DataFrame([row.model_dump() for row in rows])
different = table[table["verdict"] == ParityVerdict.DIFFERENT.value]
assert different.empty, "jede Abweichung von STplus ist erklärt oder ein Fehler"
print(len(rows), "Vergleichswerte in", table["case"].nunique(), "Fällen")
table.groupby(["origin", "verdict"]).size().unstack(fill_value=0)

In [ ]:
limits = pd.Series(detection_limits(rows)).unstack()
limits.style.format("{:.1e}")

In [ ]:
# Die Fuß-Formkreise: gearcore (Gl. (128) NB bzw. Schnittpunkt) gegen STplus, mit der eingeräumten Genauigkeit
form = table[(table["field"] == "root_form_diameter_mm") & (table["origin"] == "interface")].copy()
form["Abweichung in um"] = (form["gearcore_value"] - form["stplus_value"]).abs()
assert form["Abweichung in um"].max() <= FORM_CIRCLE_ACCURACY_MM
form[["case", "gear", "gearcore_value", "stplus_token", "Abweichung in um", "verdict"]].sort_values(
    "Abweichung in um", ascending=False
).head(8)

## 7 Grenzen dieses Inkrements und Erweiterungspunkte

Nicht unterstützt (typisierte Fehler, `extension_points.md`): Werkzeuge mit Protuberanz (DIN 3960 A.3.3), Werkzeuge mit Bearbeitungszugabe $q > 0$ (Vorverzahnung; das Fertigverzahnwerkzeug ist der Erweiterungspunkt `finishing_tool`), Werkzeuge mit vom Rad abweichendem Modul oder Profilwinkel (Erzeugungsgetriebe mit $\alpha_{\mathrm{wt0}}$, Bild 39), Schneidräder und Profilwerkzeuge, Innenverzahnung. Ein Werkzeug, das die Fußformhöhe ohne Kantenbrechwinkel angibt und dessen Fußhöhe darüber liegt, hat im Rechenkern eine unbekannte Fußrundung (Erweiterungspunkt); der `.ste`-Importer setzt dafür wie STplus den Winkel $\alpha_{\mathrm{n0}} + 10°$ (Abschnitt 8). Ein Zahn, den die Kantenbrechflanken unterhalb des Kopfkreises spitz machen, ist ein typisierter Fehler (STplus kürzt dort den Kopfkreis).

Ein Kopfkreis oberhalb der Fußlinie des Werkzeugs wird wie in STplus abgeschnitten, $d_{\mathrm{a}} = d + 2\,(x_{\mathrm{E}}\, m_{\mathrm{n}} + h_{\mathrm{fP0}})$, mit dem Hinweis `tip_circle_cut_by_tool` (Nutzerentscheidung 2026-10-03, ADR-114).

Hinweise des Ergebnisses: `no_tooth_thickness_allowance`, `root_form_diameter_by_intersection`, `tip_circle_cut_by_tool`, `edge_break_flank_not_reaching_the_tip`, `edge_break_angle_without_flank`, `edge_break_angle_not_given`, `tip_chamfer_shape_not_given`, `tip_clearance_not_positive` und die Hinweise der Paarungsgeometrie.

In [ ]:
tool = tool_from_basic_rack(iso53_basic_rack("A"))
gears = Pair(
    pinion=GearInput(
        number_of_teeth=20,
        profile_shift_coefficient=0.2,
        face_width_mm=20.0,
        tip_diameter_mm=44.8,
        tip_chamfer_radial_mm=0.0,
        tool=tool,
    ),
    wheel=GearInput(
        number_of_teeth=34,
        profile_shift_coefficient=0.1,
        face_width_mm=20.0,
        tip_diameter_mm=72.4,
        tip_chamfer_radial_mm=0.0,
        tool=tool,
    ),
)
pair = PairInput(
    normal_module_mm=2.0, normal_pressure_angle_deg=20.0, helix_angle_deg=0.0, gears=gears
)
nominal = gn.compute_generation(pair)
print("ohne Abmaße:", [w.code for w in nominal.warnings])

for label, changed in (
    ("DIN 3972 Profil III (q > 0)", din3972_tool("III", 2.0)),
    (
        "Protuberanzwerkzeug",
        tool.model_copy(update={"protuberance_mm": 0.1, "protuberance_angle_deg": 10.0}),
    ),
):
    try:
        gn.compute_generation(
            pair.model_copy(
                update={
                    "gears": Pair(
                        pinion=gears.pinion.model_copy(update={"tool": changed}), wheel=gears.wheel
                    )
                }
            )
        )
    except NotSupportedError as error:
        print(label, "->", type(error).__name__, ":", str(error)[:110], "...")

## 8 Wie STplus eine unvollständige Eingabe liest (ADR-114)

**Quelle.** STplus 11.1F, Benutzeranleitung S. 16, 19, 20, 22, 25, 186, 223–229 und 70 Probeläufe (`data/stplus_program/probes`, Vorbelegungen in `defaults.yaml`). Nutzerentscheidung 2026-10-03: Der `.ste`-Importer liest eine Datei so, wie STplus sie rechnet, und vermerkt jede Vorbelegung und jede Korrektur in `notes`; der Rechenkern bleibt explizit.

**Werkzeug.** Fehlt der Kopfhöhenfaktor, gilt 1,25, fehlt die Kopfrundung, 0,25; Fußformhöhe und Fußhöhe sind beide mit 1,3 vorbelegt. Liegt die Fußhöhe unter der Fußformhöhe, wird sie gleichgesetzt (kein Kantenbrecher). Liegt sie darüber, liegt dazwischen eine Kantenbrechflanke mit dem gegebenen Winkel, sonst $\alpha_{\mathrm{n0}} + 10°$. Zu große Werte werden begrenzt: die Kopfrundung auf die Vollrundung, die Kopfhöhe auf eine Kopfbreite des Werkzeugzahns von $0{,}120\, m_{\mathrm{n}}$, die Fußformhöhe auf eine Lückenweite von $0{,}110\, m_{\mathrm{n}}$, die Fußhöhe auf $h_{\mathrm{f0max}}^* = h_{\mathrm{FfP0}}^* + \dfrac{\pi/2 - (2\, h_{\mathrm{FfP0}}^* + 0{,}06)\tan\alpha_{\mathrm{n}}}{2\tan\alpha_{\mathrm{K}}}$ (aus den Probeläufen bestimmt, im Handbuch steht keine Formel). Die Grenzen gelten für vorbelegte wie für eingegebene Werte: das Vorbelegungswerkzeug hat bei $\alpha_{\mathrm{n}} = 28°$ die Kopfrundung 0,201, bei 30° 0,110 und die Fußformhöhe 1,265 (Probeläufe `default_hob_at_…`). Ein Kantenbrechwinkel von 90° ist das Werkzeug ohne Kantenbrecher (Handbuch S. 186): Fußhöhe gleich Fußformhöhe.

**Rad und Paarung.** Ohne Schrägungswinkel berechnet STplus ihn aus Achsabstand und $\Sigma x$ (sonst lehnt es die Eingabe ab). Ohne Kopfkreisdurchmesser gilt $d_{\mathrm{a}} = d + 2\, m_{\mathrm{n}} (1 + x)$ ohne Kopfhöhenänderung. Ist nur das obere Zahnweitenabmaß gegeben, wird das untere gleichgesetzt. Die Abmaßreihe c25, die STplus vorbelegt, folgt mit Inkrement 5 (DIN 3967).

**Wo Handbuch und Programm sich widersprechen** (am FZG bekannt), gilt das Programm:

| Stelle | Handbuch | Programm 11.1F |
|---|---|---|
| S. 224 | $h_{\mathrm{Ff0}}^*$ mit 1,1 vorbelegt | 1,300 (auch im Beispiel des Handbuchs, S. 259); die Schlüssel `VB_FUSS…` wirken nicht |
| S. 186 | $h_{\mathrm{f0}} < h_{\mathrm{Ff0}}$ wird zu $h_{\mathrm{f0max}}$ | $h_{\mathrm{f0}} = h_{\mathrm{Ff0}}$ |
| S. 225 | max. Kopfkantenbruchfaktor 0,02 | begrenzt auf $0{,}20\, m_{\mathrm{n}}$ |
| S. 224 | die Steuergrößen $s_{\mathrm{a0}}^*$ und $e_{\mathrm{Ff0}}^*$, die Werkzeugkopfhöhe und Fußformhöhe begrenzen, sind mit 0,2 und 0,4 vorbelegt | das Programm belegt sie mit 0,12 und 0,11 vor; eingegeben wirken sie wie beschrieben |
| S. 224 | $e_{\mathrm{f0}}^*$ ist eine Lückenweite, vorbelegt mit 0,06 | Lückenweite an der Fußlinie $2\, e_{\mathrm{f0}}^* \tan\alpha_{\mathrm{n}}$; ohne Eingabe $e_{\mathrm{f0}}^* = 0{,}03$ |
| S. 223, 225 | Steuergrößen lassen sich „zwischen“ den Grenzen ihres Einstellbereichs variieren | die Grenzen selbst werden nicht angenommen |

**Was der Importer nicht übersetzt**, meldet er als typisierten Fehler statt einer stillen Annahme: die fünf anderen Festlegungen des Kopfkreises (Handbuch Bild 4.12: Kopfdicke, Kopfkreis durch das Werkzeug, Kopfhöhenänderung nach DIN 3960, Kopfspielfaktor, Kopfhöhenfaktor des Bezugsprofils) und die kreisförmige statt der elliptischen Werkzeugkopfrundung. Die drei Steuergrößen der Werkzeuggrenzen liest er. Eine Steuergröße ist ein Wert für die ganze Stufe: der erste Wert gilt für beide Räder (Probeläufe `tip_chamfer_tangential_two_values`, `controls_first_value_holds`); liegt er außerhalb seines Einstellbereichs, bleibt die Vorbelegung (Handbuch S. 223). Einen Kantenbrechwinkel zwischen 85° und 90° rechnet STplus nicht mehr (Abbruch bei 88°); der Importer lehnt ihn ab.

In [ ]:
# Probelauf: Fußformhöhe 0,9 und Fußhöhe 1,5 ohne Kantenbrechwinkel; Kopfkreis 46,4 mm
from gearcore.data import data_path
from gearcore.io.ste import load_ste, pair_input_from_ste
from gearcore.stplus_program import probe_listing

imported = pair_input_from_ste(
    load_ste(data_path("stplus_program", "probes", "tool_root_both_heights", "input.ste"))
)
for note in imported.notes:
    print("-", note)

tool = imported.pair.gears.pinion.tool
assert math.isclose(tool.edge_break_angle_deg, 30.0)
remark = [
    line.strip()
    for line in probe_listing("tool_root_both_heights").splitlines()
    if "alfa_K0" in line
]
print(remark)
assert remark == ["Kopfkantenbruch an Rad 1 durch Werkzeug (alfa_K0 = 30.00 grd)"]

In [ ]:
# ein Kopfkreis oberhalb der Werkzeug-Fußlinie wird abgeschnitten (Probelauf: Fußformhöhe 0,9, Fußhöhe 0,5
# -> beide 0,9; Kopfkreis 46,4 mm). STplus druckt 45,021 mm, weil es mit der Abmaßreihe c25 erzeugt
# (x_E = 0,3554); ohne Abmaß (x_E = x = 0,4) schneidet das Werkzeug bei 45,2 mm.
probe = "tool_root_dedendum_below_form_height"
imported = pair_input_from_ste(load_ste(data_path("stplus_program", "probes", probe, "input.ste")))
result = gn.compute_generation(imported.pair)
print(
    "gegeben:",
    imported.pair.gears.pinion.tip_diameter_mm,
    "mm, erzeugt:",
    result.gears.pinion.tip_diameter_mm,
    "mm",
)
print([w.code for w in result.gears.pinion.warnings])
remark = [line.strip() for line in probe_listing(probe).splitlines() if "geschnitten" in line]
print(remark)
assert "tip_circle_cut_by_tool" in [w.code for w in result.gears.pinion.warnings]
assert result.gears.pinion.tip_diameter_mm < imported.pair.gears.pinion.tip_diameter_mm